# UTS Data Acquisition & Management: Akuisisi Data Daftar ODTW Kabupaten Majalengka

**Topik:** Pariwisata (Objek Daya Tarik Wisata, Kabupaten Majalengka, Jawa Barat)
**Dosen:** Dr. I Gede Totok Suryawan, S.Kom., M.T. | **Kelas:** E | **Prodi:** Informatika, INSTIKI

**Tiga sumber akuisisi:**
1. **Berkas**: dataset ODTW dari data.go.id (ada di `data/raw/`)
2. **API**: Open-Meteo, cuaca harian di pusat Kabupaten Majalengka (tanpa API key)
3. **Scraping**: satu tabel kecamatan dari halaman Wikipedia Indonesia, setelah cek `robots.txt`

**Cara menjalankan:** Runtime > Run all. Semua sel harus dijalankan sebelum dikumpulkan.

In [ ]:
# ============================================================
# 0. SETUP: library, konfigurasi, dan folder kerja
# ============================================================
import os, json, time, hashlib, datetime as dt, urllib.robotparser
from io import StringIO
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Email kampus wajib untuk User-Agent (syarat soal). GANTI dengan email kampus Anda.
CONTACT_EMAIL = "GANTI_email_kampus@instiki.ac.id"
USER_AGENT = f"UTS-DAM-Informatika-E/1.0 (mahasiswa; kontak: {CONTACT_EMAIL})"
HEADERS = {"User-Agent": USER_AGENT}
TIMEOUT = 20   # detik, wajib pada setiap request
DELAY = 2      # jeda antar-request (detik)

# Repo dikloning ke /content/repo
REPO_DIR = "/content/repo"
RAW_DIR = os.path.join(REPO_DIR, "data", "raw")
PROC_DIR = os.path.join(REPO_DIR, "data", "processed")
DOCS_DIR = os.path.join(REPO_DIR, "docs")
for d in [RAW_DIR, PROC_DIR, DOCS_DIR]:
    os.makedirs(d, exist_ok=True)

# Sesi HTTP dengan retry otomatis untuk status 429 dan 5xx
session = requests.Session()
session.headers.update(HEADERS)
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])))

def md5_of(path):
    # Hitung hash MD5 berkas untuk catatan provenance
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

print("Folder siap. Repo:", REPO_DIR)

In [ ]:
# Kloning repo jika belum ada. Ganti USERNAME/NAMA-REPO dengan repo Anda (repo publik).
if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    !git clone https://github.com/USERNAME/NAMA-REPO.git /content/repo
print(os.listdir(RAW_DIR))

---
## Soal 1: Baca berkas, identifikasi data, dan estimasi storage

Parameter pembacaan:
- `sep=","` dan `encoding="utf-8"`: CSV dengan pemisah koma dan UTF-8. Akhir baris CRLF dibaca otomatis.
- `dtype=str`: semua kolom dibaca sebagai teks agar kode wilayah (misalnya `3210050` dan `32.10.05`) tidak berubah menjadi angka.
- `na_values=["", "-"]`: string kosong dan `-` dianggap nilai hilang.

In [ ]:
# Baca berkas mentah (tidak diubah) dengan parameter yang tepat
ODTW_PATH = os.path.join(RAW_DIR, "dftr_ptns_bjk_dy_trk_wst_dtw_d_kbptn_mjlngk_2.csv")
odtw = pd.read_csv(ODTW_PATH, sep=",", encoding="utf-8", dtype=str,
                   na_values=["", "-"], keep_default_na=True)

# Konversi tipe setelah pembacaan
odtw["id"] = odtw["id"].astype(int)
odtw["tahun"] = odtw["tahun"].astype(int)

print("Bentuk data (baris, kolom):", odtw.shape)
print("\nTipe data:")
print(odtw.dtypes)

In [ ]:
# Identifikasi dataset: jenis, ukuran, cakupan, satuan, dan estimasi storage
size_bytes = os.path.getsize(ODTW_PATH)
rows, cols = odtw.shape

print("Jenis data        : terstruktur (tabel baris-kolom dengan tipe konsisten)")
print("Jumlah baris      :", rows)
print("Jumlah kolom      :", cols)
print("Cakupan wilayah   :", odtw["bps_nama_kabupaten_kota"].unique()[0],
      "| kecamatan:", odtw["bps_nama_kecamatan"].nunique())
print("Cakupan waktu     : tahun data", odtw["tahun"].value_counts().sort_index().to_dict())
print("Satuan            :", odtw["satuan"].unique().tolist(), "(setiap baris adalah satu lokasi wisata)")
print("Missing per kolom :", odtw.isna().sum().to_dict())
print("Ukuran berkas     :", size_bytes, "byte")

# Estimasi storage 1 tahun = ukuran berkas x 12 bulan, ditambah cadangan 30%
# Asumsi: setiap bulan disimpan snapshot dengan ukuran yang sama
storage_1y = size_bytes * 12
storage_total = storage_1y * 1.30
print(f"\nEstimasi storage 1 tahun        : {storage_1y:,.0f} byte ({storage_1y/1024:,.1f} KiB)")
print(f"Estimasi + cadangan 30%         : {storage_total:,.0f} byte ({storage_total/1024:,.1f} KiB)")

**Asumsi estimasi storage:** (1) ukuran snapshot bulanan sama dengan berkas saat ini; (2) tidak ada versi tambahan atau hasil olahan yang disimpan di luar `data/raw`; (3) cadangan 30% untuk backup dan variasi ukuran.

---
## Soal 2: Lisensi, PII, dan quasi-identifier

Lisensi dataset di halaman data.go.id **belum diketahui**. Status ini dicatat sebagai temuan di `docs/provenance.csv` dan `docs/ethics_checklist.md`, bukan diasumsikan. Pemeriksaan kolom PII dan quasi-identifier dilakukan di sel berikut.

In [ ]:
# Klasifikasi setiap kolom: PII, quasi-identifier, atau non-PII
klasifikasi = {
    "id": "non-PII (nomor urut internal)",
    "kode_provinsi": "non-PII (kode wilayah)",
    "nama_provinsi": "non-PII (wilayah)",
    "bps_kode_kabupaten_kota": "non-PII (kode wilayah)",
    "bps_nama_kabupaten_kota": "non-PII (wilayah)",
    "bps_kode_kecamatan": "quasi-identifier (level kecamatan)",
    "bps_nama_kecamatan": "quasi-identifier (level kecamatan)",
    "kemendagri_kode_kecamatan": "non-PII (kode wilayah)",
    "kemendagri_nama_kecamatan": "non-PII (wilayah)",
    "nama_odtw": "non-PII (nama objek wisata publik)",
    "alamat": "quasi-identifier (level desa, bukan alamat rumah tangga)",
    "satuan": "non-PII",
    "tahun": "non-PII",
}
for col in odtw.columns:
    print(f"{col:28s} -> {klasifikasi.get(col, 'belum diklasifikasi')}")

# Cek pola PII umum (email, nomor telepon, NIK 16 digit) di seluruh kolom
pola = {"email": r"[\w.+-]+@[\w-]+\.[\w.]+", "telepon": r"(?:\+62|08)\d{8,}", "nik16": r"\b\d{16}\b"}
hasil = {nama: int(odtw.apply(lambda s: s.astype(str).str.contains(p, regex=True)).sum().sum())
         for nama, p in pola.items()}
print("\nHasil pencarian pola PII:", hasil)
print("Kesimpulan: tidak ditemukan PII langsung (email, telepon, NIK).")

---
## Soal 3: Akuisisi API (Open-Meteo)

Portal data.go.id tidak menyediakan API untuk dataset ini, sehingga dipakai **Open-Meteo** (API publik, tanpa key) untuk cuaca di Kabupaten Majalengka. Koordinat yang dipakai adalah **perkiraan pusat wilayah**.

In [ ]:
OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": -6.8363,       # perkiraan pusat Kabupaten Majalengka
    "longitude": 108.2277,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "past_days": 31,           # 31 hari ke belakang + 1 hari ke depan = 32 baris
    "forecast_days": 1,
    "timezone": "Asia/Jakarta",
}

# Loop percobaan manual sambil memeriksa status code
resp = None
for attempt in range(1, 4):
    try:
        resp = session.get(OPEN_METEO_URL, params=params, timeout=TIMEOUT)
        print(f"Percobaan {attempt}: status {resp.status_code}")
        if resp.status_code == 200:
            break
    except requests.RequestException as e:
        print(f"Percobaan {attempt}: error jaringan -> {e}")
    time.sleep(DELAY * attempt)   # jeda bertambah setiap gagal

if resp is None or resp.status_code != 200:
    raise RuntimeError("API gagal setelah 3 percobaan. Periksa koneksi atau status layanan.")

time.sleep(DELAY)  # jeda setelah request

# Simpan respons JSON mentah tanpa diubah
api_raw_path = os.path.join(RAW_DIR, "api_openmeteo_majalengka.json")
with open(api_raw_path, "w", encoding="utf-8") as f:
    f.write(resp.text)

# Normalisasi JSON menjadi tabel
payload = resp.json()
weather = pd.DataFrame(payload["daily"]).rename(columns={"time": "tanggal"})
weather["tanggal"] = pd.to_datetime(weather["tanggal"])
weather["lokasi"] = "Kabupaten Majalengka (pusat, perkiraan)"
weather["sumber"] = "api_openmeteo"
assert len(weather) >= 30, "Tabel API harus minimal 30 baris"

weather.to_csv(os.path.join(PROC_DIR, "api_cuaca_majalengka.csv"), index=False)
print("Baris API:", len(weather))
weather.head()

---
## Soal 4: Scraping halaman statis yang diizinkan

Sumber: halaman **Wikipedia Indonesia** tentang Kabupaten Majalengka. Sebelum mengambil halaman, `robots.txt` diperiksa dan hasilnya disimpan sebagai bukti kepatuhan.

In [ ]:
SCRAPE_URL = "https://id.wikipedia.org/wiki/Kabupaten_Majalengka"
ROBOTS_URL = "https://id.wikipedia.org/robots.txt"

# 1) Periksa robots.txt
r = session.get(ROBOTS_URL, timeout=TIMEOUT)
print("Status robots.txt:", r.status_code)
rp = urllib.robotparser.RobotFileParser()
rp.parse(r.text.splitlines())
boleh = rp.can_fetch(USER_AGENT, SCRAPE_URL)
print("Boleh mengambil", SCRAPE_URL, "->", boleh)

# Simpan bukti pemeriksaan robots.txt
with open(os.path.join(DOCS_DIR, "robots_check.txt"), "w", encoding="utf-8") as f:
    f.write(f"Waktu pemeriksaan : {dt.datetime.now().isoformat()}\n")
    f.write(f"robots.txt URL    : {ROBOTS_URL} (status {r.status_code})\n")
    f.write(f"Halaman target    : {SCRAPE_URL}\n")
    f.write(f"User-Agent        : {USER_AGENT}\n")
    f.write(f"Hasil can_fetch   : {boleh}\n")
    f.write("Baris user-agent dan disallow dari robots.txt:\n")
    f.write("\n".join(l for l in r.text.splitlines() if l.lower().startswith(("user-agent", "disallow"))))
assert boleh, "robots.txt melarang pengambilan halaman ini. Pilih halaman lain."

# 2) Ambil halaman dengan timeout dan jeda
time.sleep(DELAY)
page = session.get(SCRAPE_URL, timeout=TIMEOUT)
page.raise_for_status()
with open(os.path.join(RAW_DIR, "wiki_kabupaten_majalengka.html"), "w", encoding="utf-8") as f:
    f.write(page.text)

# 3) Ekstrak tabel dengan pd.read_html, pilih tabel yang namanya cocok dengan kecamatan di dataset
kec_dataset = set(odtw["bps_nama_kecamatan"].str.upper().str.strip())
tables = pd.read_html(StringIO(page.text))
print("Jumlah tabel di halaman:", len(tables))

best_idx, best_col, best_score = None, None, 0
for i, t in enumerate(tables):
    t.columns = [" ".join(map(str, c)).strip() if isinstance(c, tuple) else str(c) for c in t.columns]
    for col in t.columns:
        skor = len(set(t[col].astype(str).str.upper().str.strip()) & kec_dataset)
        if skor > best_score:
            best_idx, best_col, best_score = i, col, skor

assert best_score >= 10, "Tabel kecamatan tidak ditemukan. Periksa halaman sumber."
kec_raw = tables[best_idx]
print(f"Tabel dipilih: indeks {best_idx}, kolom kunci '{best_col}', cocok {best_score} kecamatan")

# Simpan tabel mentah tanpa perubahan lebih lanjut
kec_raw_path = os.path.join(RAW_DIR, "wiki_tabel_kecamatan.csv")
kec_raw.to_csv(kec_raw_path, index=False)
KEC_COL = best_col
kec_raw.head()

---
## Soal 5: Penggabungan data, validasi, dan bukti tanpa duplikasi

Penggabungan dilakukan antara **berkas** (ODTW) dan **scraping** (tabel kecamatan) melalui kunci nama kecamatan. Relasinya **banyak ODTW ke satu kecamatan**, sehingga `validate="many_to_one"` dipakai.

In [ ]:
# Normalisasi kunci penggabungan
odtw["kunci_kecamatan"] = odtw["bps_nama_kecamatan"].str.upper().str.strip()
kec = pd.read_csv(kec_raw_path, dtype=str)
kec["kunci_kecamatan"] = kec[KEC_COL].str.upper().str.strip()
kec = kec.drop_duplicates("kunci_kecamatan")   # sisi 'satu' harus unik
assert kec["kunci_kecamatan"].is_unique

# Penggabungan dengan validasi relasi many-to-one
merged = odtw.merge(kec, on="kunci_kecamatan", how="left",
                    validate="many_to_one", suffixes=("", "_wiki"), indicator=True)

print("Baris ODTW         :", len(odtw))
print("Baris hasil gabung :", len(merged))
print("Tanpa pasangan kecamatan:", int((merged["_merge"] == "left_only").sum()))

In [ ]:
# BUKTI TIDAK ADA DUPLIKASI
# (a) Jumlah baris tidak bertambah akibat merge
assert len(merged) == len(odtw), "Merge menambah baris: ada duplikasi kunci"

# (b) Setiap id unik
assert merged["id"].is_unique, "Ada id yang muncul lebih dari sekali"

# (c) Baris dengan isi identik selain id dan tahun (duplikasi 'semantik' antar-tahun)
kolom_isi = [c for c in odtw.columns if c not in ["id", "tahun", "kunci_kecamatan"]]
dup_isi = odtw[odtw.duplicated(subset=kolom_isi, keep=False)]
print("Baris dengan isi identik (selain id dan tahun):", len(dup_isi))
print(dup_isi[["id", "nama_odtw", "alamat", "tahun"]].to_string(index=False))
print("Catatan: baris di atas identik kecuali tahun. Dicatat, tidak dihapus.")

# (d) Pasangan nama sama di kecamatan sama tetapi isi berbeda: dicatat, tidak dihapus
dup_nama = odtw[odtw.duplicated(["nama_odtw", "bps_nama_kecamatan"], keep=False)]
print("\nPasangan nama+kecamatan sama (id berbeda, dicatat):", len(dup_nama))

**Catatan temuan:** `TALAGA PANCAR` (id 209 dan 243) memiliki alamat dan kecamatan yang sama, hanya berbeda tahun (2022 dan 2023). Ini kemungkinan pencatatan ulang di tahun berbeda, bukan duplikasi teknis. Baris dipertahankan karena data mentah tidak boleh diubah. Pada analisis lanjutan, baris tersebut dapat difilter berdasarkan tahun.

In [ ]:
# Simpan hasil penggabungan (tanpa kolom indikator internal)
merged_out = merged.drop(columns=["_merge"])
out_path = os.path.join(PROC_DIR, "odtw_kecamatan_gabung.csv")
merged_out.to_csv(out_path, index=False)

# Metadata hasil penggabungan
metadata = {
    "dibuat": dt.datetime.now().isoformat(),
    "sumber_gabungan": ["data/raw/dftr_ptns_bjk_dy_trk_wst_dtw_d_kbptn_mjlngk_2.csv",
                        "data/raw/wiki_tabel_kecamatan.csv"],
    "kunci": "kecamatan (huruf besar, tanpa spasi berlebih)",
    "validasi": "many_to_one",
    "baris_input_odtw": int(len(odtw)),
    "baris_output": int(len(merged_out)),
    "baris_tanpa_pasangan": int((merged["_merge"] == "left_only").sum()),
    "kolom_output": list(merged_out.columns),
    "md5_output": md5_of(out_path),
}
with open(os.path.join(PROC_DIR, "metadata_gabungan.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(json.dumps({k: v for k, v in metadata.items() if k != "kolom_output"}, ensure_ascii=False, indent=2))

---
## Provenance dan Data Ethics Checklist

Sel berikut membuat `docs/provenance.csv` berisi URL, tanggal akses, lisensi, dan hash MD5 untuk setiap berkas mentah. Isi Data Ethics Checklist ada di `docs/ethics_checklist.md`.

In [ ]:
# Catatan provenance untuk setiap sumber mentah
hari_ini = dt.date.today().isoformat()
sumber = [
    ("berkas", "https://data.go.id/dataset/dataset/daftar-potensi-objek-daya-tarik-wisata-odtw-di-kabupaten-majalengka",
     hari_ini, "tidak diketahui (perlu dicek di halaman dataset)",
     "data/raw/dftr_ptns_bjk_dy_trk_wst_dtw_d_kbptn_mjlngk_2.csv", ODTW_PATH),
    ("api", OPEN_METEO_URL, hari_ini, "Open-Meteo: CC BY 4.0 (verifikasi di open-meteo.com)",
     "data/raw/api_openmeteo_majalengka.json", api_raw_path),
    ("scraping", SCRAPE_URL, hari_ini, "CC BY-SA 4.0 (Wikipedia)",
     "data/raw/wiki_kabupaten_majalengka.html", os.path.join(RAW_DIR, "wiki_kabupaten_majalengka.html")),
    ("scraping-tabel", SCRAPE_URL, hari_ini, "CC BY-SA 4.0 (Wikipedia)",
     "data/raw/wiki_tabel_kecamatan.csv", kec_raw_path),
]
prov = pd.DataFrame([{
    "jenis": j, "url": u, "tanggal_akses": t, "lisensi": l, "berkas": b,
    "md5": md5_of(p)} for j, u, t, l, b, p in sumber])
prov.to_csv(os.path.join(DOCS_DIR, "provenance.csv"), index=False)
prov

In [ ]:
# Dorong hasil ke GitHub dari Colab.
# Token disimpan di Colab Secrets (ikon kunci, nama GITHUB_TOKEN), BUKAN di notebook.
from google.colab import userdata
GH_TOKEN = userdata.get("GITHUB_TOKEN")
GH_REPO = "USERNAME/NAMA-REPO"   # ganti sesuai repo Anda

os.environ["GH_TOKEN"] = GH_TOKEN
os.environ["GH_REPO"] = GH_REPO
!cd /content/repo && git config user.name "Mahasiswa Informatika" && git config user.email "{CONTACT_EMAIL}" && git add -A && (git commit -m "UTS DAM: akuisisi berkas, API, scraping, dan penggabungan" || echo "Tidak ada perubahan")
!cd /content/repo && git push https://$GH_TOKEN@github.com/$GH_REPO.git HEAD:main